In [1]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import netCDF4 as n
import glob
import xarray as xr
import seaborn as sns
import matplotlib.lines as mlines
import pandas as pd
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.stats import linregress
from collections import Counter
sns.set_context("paper")
sns.set_style("whitegrid")
computed_name = 'ComputedScalarsHelene'
mnt_pth = '/Volumes/FMac/computing/data/antarctica/' #mount path
pth_ismip6hack= mnt_pth + 'ismip6/hackathon/ComputedScalars/' #write folder
pth_ismip6 =mnt_pth + 'ismip6/hackathon/' #read folder
pth_helene =mnt_pth + 'ismip6/hackathon/ComputedScalarsHelene/'
pth_table = '/Users/fmcc0003/issm/projects/ISMIP6Hackathon/GroupBMBtodSLR/data_analysis/tables/'

In [2]:
# choose variable,factor and path
var_area = 'iareafl'
var_bmb = 'shelfmelt'

var_dslc = 'dslc_anom'
pth_var_dslc = 'dslc_anom_jb'
factor = -1

fpath = f'{mnt_pth}/ismip6/hackathon/figures/recalc_{var_dslc}_dslcs_dils/'
fpathscat = f'{mnt_pth}/ismip6/hackathon/figures/recalc_{var_dslc}_dslcs_dils_scatter/'
fname = f'cumulative_{var_dslc}_'

In [3]:
expnames = ['expAE02', 'expAE03', 'expAE04', 'expAE05']
removeFileID = []

renamer = {}
renamer['DOE_MALI_4km']='DOE_MALI_1'
renamer['DOE_MALI_8km_Ant95']='DOE_MALI_2'
renamer['DOE_MALI_8km_AntMean']='DOE_MALI_3'

metadata_file = 'Metadata.txt'
def get_model(string,exp):
    return string.split('/')[-1].split('AIS_')[-1].split('_'+str(exp))[0]

# Generate loop_info DataFrame
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_ismip6hack+f'{exp}/{pth_var_dslc}/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
df['AIS']=0
df_org = df.copy()

metadata_file = '/Users/fmcc0003/issm/projects/ISMIP6Hackathon/GroupTFtoOcean/data_analysis/Metadata_icefront.txt'
df_info = pd.read_csv(metadata_file)
df_dils = pd.read_csv('./tables/linear_dslc_sensitivity_from_cumBMB2.csv')
df_dils_n = pd.read_csv('./tables/linear_dslc_sensitivity_from_cumBMB_intercept2.csv')

In [4]:
df_dils

,Experiment,Model,Path,sector_1,sector_2,sector_3,sector_4,sector_5,sector_6,sector_7,...,sector_18,AIS,region_1,region_2,region_3,Amundsen,Ross,FilchnerRonne,Aurora,Wilkes
0,expAE02,DC_ISSM,/Volumes/FMac/computing/data/antarctica/ismip6...,8.890075e-07,0.000003,0.000003,0.000004,0.000005,3.196344e-07,0.000005,...,0.000003,0.000003,0.000004,0.000003,-3.112771e-07,0.000004,0.000002,0.000004,0.000005,0.000005
1,expAE02,DOE_MALI_4km,/Volumes/FMac/computing/data/antarctica/ismip6...,9.958385e-06,0.000006,0.000007,0.000017,0.000008,6.500727e-07,0.000020,...,0.000010,0.000009,0.000009,0.000008,7.253234e-06,0.000017,0.000005,0.000008,0.000014,0.000020
2,expAE02,DOE_MALI_8km_Ant95,/Volumes/FMac/computing/data/antarctica/ismip6...,1.085729e-05,0.000011,0.000008,0.000014,0.000010,6.551747e-07,0.000026,...,0.000009,0.000011,0.000012,0.000010,6.605011e-06,0.000014,0.000008,0.000009,0.000016,0.000026
3,expAE02,DOE_MALI_8km_AntMean,/Volumes/FMac/computing/data/antarctica/ismip6...,9.829671e-06,0.000009,0.000008,0.000015,0.000009,6.167189e-07,0.000025,...,0.000010,0.000010,0.000011,0.000009,6.659000e-06,0.000015,0.000007,0.000009,0.000015,0.000025
4,expAE02,IGE_ElmerIce,/Volumes/FMac/computing/data/antarctica/ismip6...,8.384436e-07,0.000002,0.000002,0.000003,0.000004,3.688361e-07,0.000008,...,0.000002,0.000003,0.000003,0.000003,-1.106049e-06,0.000003,0.000001,0.000003,0.000005,0.000008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167,expAE05,VUW_PISM2,/Volumes/FMac/computing/data/antarctica/ismip6...,1.362071e-05,0.000012,0.000013,0.000030,0.000011,-3.125696e-07,0.000022,...,0.000013,0.000014,0.000015,0.000012,8.224060e-06,0.000030,0.000010,0.000010,0.000014,0.000022
168,expAE05,VUW_PISM2_s1,/Volumes/FMac/computing/data/antarctica/ismip6...,1.313614e-05,0.000015,0.000014,0.000034,0.000012,-5.700851e-07,0.000024,...,0.000015,0.000015,0.000015,0.000014,1.018034e-05,0.000034,0.000012,0.000011,0.000015,0.000024
169,expAE05,VUW_PISM2_s2,/Volumes/FMac/computing/data/antarctica/ismip6...,1.451278e-05,0.000011,0.000012,0.000026,0.000011,8.643503e-07,0.000019,...,0.000012,0.000013,0.000015,0.000011,6.799766e-06,0.000026,0.000009,0.000010,0.000014,0.000019
170,expAE05,VUW_PISM2_s3,/Volumes/FMac/computing/data/antarctica/ismip6...,1.272611e-05,0.000009,0.000011,0.000023,0.000010,1.368207e-06,0.000019,...,0.000010,0.000011,0.000011,0.000011,5.342968e-06,0.000023,0.000007,0.000009,0.000011,0.000019


In [5]:
regions = ['AIS', 'Amundsen', 'Ross', 'FilchnerRonne', 'Aurora', 'Wilkes']

dic_area_of_interest ={}
dic_area_of_interest['AIS']= ['']
dic_area_of_interest['WestAntarctica'] = ['_region_1']
dic_area_of_interest['EastAntarctica'] = ['_region_2']
dic_area_of_interest['Peninsula'] = ['_region_3']
dic_area_of_interest['Amundsen'] = ['_sector_4']
dic_area_of_interest['FilchnerRonne'] = ['_sector_5','_sector_11']
dic_area_of_interest['Ross'] = ['_sector_2','_sector_6']
dic_area_of_interest['Aurora'] = ['_sector_8']
dic_area_of_interest['Wilkes'] = ['_sector_7']

In [6]:
df_dils['WestAntarctica'] = df_dils['region_1']
df_dils['EastAntarctica'] = df_dils['region_2']
df_dils['Peninsula'] = df_dils['region_3']
df_dils_n['WestAntarctica'] = df_dils_n['region_1']
df_dils_n['EastAntarctica'] = df_dils_n['region_2']
df_dils_n['Peninsula'] = df_dils_n['region_3']

In [7]:
area = False # better without area

if os.path.exists(fpath):
    print( f"The path '{fpath}' already exists.")
else:
    os.makedirs(fpath)
    print( f"The path '{fpath}' did not exist, so it has been created.")
if os.path.exists(fpathscat):
    print( f"The path '{fpathscat}' already exists.")
else:
    os.makedirs(fpathscat)
    print( f"The path '{fpathscat}' did not exist, so it has been created.")

dic_all_dils = {}
dic_all_dils_intercept = {}
dic_all_err = {}

for key in dic_area_of_interest.keys():
    dic_key_prmse = {}
    dic_key_dils = {}
    dic_key_dils_intercept = {}
    lst = dic_area_of_interest[key]

    a4_width = 8.27  # inches
    a4_height = 11.69
    colors = ['#1f77b4','#2ca02c','#9467bd','#e377c2']
    fs = 8

    df = df_org[df_org['Experiment']==exp].reset_index(drop=True)
    for j,exp in enumerate(expnames):
        dic_prmse ={}
        dic_dils ={}
        dic_dils_intercept ={}
        f,ax = plt.subplots(int(np.ceil(df.shape[0]/2)),2,figsize=(a4_width, a4_height))
        fscat,axscat = plt.subplots(int(np.ceil(df.shape[0]/2)),2,figsize=(a4_width, a4_height))
        
        df = df_org[df_org['Experiment']== exp].reset_index(drop=True)
        dils_exp =df_dils[df_dils['Experiment']== exp].reset_index(drop=True)
        dils_n_exp =df_dils_n[df_dils_n['Experiment']== exp].reset_index(drop=True)

        for i in range (df.shape[0]):
            ix =i //2
            iy=i%2
            pth_way = df.iloc[i].Path
            nmname = df.iloc[i].Model
            
            #dslc
            pth_dslc = f'{pth_ismip6hack}/{df.iloc[i].Experiment}/{pth_var_dslc}/computed_{var_dslc}_AIS_{nmname}_{df.iloc[i].Experiment}.nc'
            d3 = xr.open_dataset(pth_dslc,decode_times=False )
        
            #shelf area
            pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/{var_area}/computed_{var_area}_AIS_{nmname}_{df.iloc[i].Experiment}.nc'
            d2 = xr.open_dataset(pth_area,decode_times=False )
            
            #bmb
            pth_bmb = f'{pth_helene}/{df.iloc[i].Experiment}/{var_bmb}/computed_{var_bmb}_AIS_{nmname}_{df.iloc[i].Experiment}.nc'
            d4 = xr.open_dataset(pth_bmb,decode_times=False )
            tmin = np.min([d2.time.size,d3.time.size,d4.time.size])
            
            the_ys = np.zeros((len(lst),tmin))
            the_xs = np.zeros((len(lst),tmin))
            the_areas = np.zeros((len(lst),tmin))
            for k,nm in enumerate(lst):
                the_ys[k,:]= -d3[f'{var_dslc}{nm}'].values[:tmin].copy()
                the_areas[k,:]=d2[f'{var_area}{nm}'].values[:tmin]
                if area:
                    the_xs[k,:]= d4[f'{var_bmb}{nm}'].values[:tmin].copy()*factor*the_areas[k,:]
                else:
                    the_xs[k,:]= d4[f'{var_bmb}{nm}'].values[:tmin].copy()*factor

            x = np.cumsum(np.sum(the_xs, axis=0)).squeeze()
            y = np.sum(the_ys, axis=0)
            dils, dils_n, r_value, p_value, std_err = linregress(x, y)
            dic_dils[nmname] = dils.copy()
            dic_dils_intercept[nmname] = dils_n.copy()
            
            ax2 = ax[ix,iy]            
            ax2.plot(d3.time[:tmin],y ,color = colors[j],label = f'{exp}')
            ax2.plot(d3.time[:tmin],dils*x+dils_n,color = 'k',linestyle ='dashed',label = f'recalc')

            yfit = dils_n + dils * x
            axscat2 = axscat[ix,iy]            
            axscat2.scatter(x, y, color = colors[j], label = f'{exp}')
            axscat2.plot(x, yfit, color = 'k')
            
            err = (yfit - y)**2
            
            dilsk = np.isnan(err) | (y == 0)
            rmse = (np.sum(err[~dilsk])/err[~dilsk].size)**0.5
            prmse = (( (1/err[~dilsk].size)*np.sum(err[~dilsk]))/np.sum(y[~dilsk]**2))**0.5 *100
            dic_prmse[nmname] = prmse.copy()

            if df.iloc[i].Model in ['DOE_MALI_4km', 'DOE_MALI_8km_Ant95', 'DOE_MALI_8km_AntMean']:
                nmname = renamer[df.iloc[i].Model]

            df_info_model = df_info[df_info['fileID']==nmname]
            df_info_model_exp = df_info_model[df_info_model['Experiment']==exp]
            meltparam =  df_info_model_exp['Melt parameterisation'].values[0]
            icefront =  df_info_model_exp['ice front'].values[0]
            
            ax2.text(0.02, 0.98,f' Model {nmname},\n {meltparam}, {icefront} ',transform=ax2.transAxes,fontsize = fs, va='top',color = 'red')
            axscat2.text(0.02, 0.98,f' Model {nmname},\n {meltparam}, {icefront} ',transform=axscat2.transAxes,fontsize = fs, va='top',color = 'red')

        dic_key_prmse[exp]=dic_prmse.copy()
        dic_key_dils[exp]=dic_dils.copy()
        dic_key_dils_intercept[exp]=dic_dils_intercept.copy()
        
        handles, labels = ax[0, 0].get_legend_handles_labels()
        f.legend(handles, labels, loc='upper right', title='Experiments')
        handles, labels = ax2.get_legend_handles_labels()
        f.legend(handles, labels, loc='upper left', title='Experiments')
        f.suptitle(f'{fname} {key} ice shelf(GT/yr)',fontsize = 12)
        if area:
            f.savefig(f'{fpath}{fname}{key}_{exp}.pdf')
            plt.close(f)
        else:
            f.savefig(f'{fpath}{fname}{key}_{exp}_noarea.pdf')
            plt.close(f)

        hscat, lscat = axscat[0, 0].get_legend_handles_labels()
        fscat.legend(hscat, lscat, loc='upper right', title='Experiments')
        hscat, lscat = axscat2.get_legend_handles_labels()
        fscat.legend(hscat, lscat, loc='upper left', title='Experiments')
        fscat.suptitle(f'{fname} {key} ice shelf (GT/yr)',fontsize = 12)
        fscat.savefig(f'{fpathscat}{fname}{key}_{exp}_noarea.pdf')
        plt.close(fscat)
    dic_all_dils[key] = dic_key_dils.copy()
    dic_all_dils_intercept[key] = dic_key_dils_intercept.copy()
    dic_all_err[key] = dic_key_prmse.copy()

The path '/Volumes/FMac/computing/data/antarctica//ismip6/hackathon/figures/recalc_dslc_anom_dslcs_dils/' already exists.
The path '/Volumes/FMac/computing/data/antarctica//ismip6/hackathon/figures/recalc_dslc_anom_dslcs_dils_scatter/' already exists.


In [11]:
## save dils to csv
df_dils_new = df_dils.copy()
df_dils_new=df_dils_new.drop(df_dils_new.columns[3:21],axis=1)
df_dils_new=df_dils_new.drop(df_dils_new.columns[4:7],axis=1)
for key in dic_area_of_interest.keys():
    df_dils_new[key]=0

for i in range (df_dils_new.shape[0]):
    model = df_dils_new.iloc[i].Model
    exp = df_dils_new.iloc[i].Experiment
    for j,key in enumerate(dic_area_of_interest.keys()):
        df_dils_new.loc[i,key]= dic_all_dils[key][exp][model].copy()
df_dils_new.to_csv(pth_table + '/linear_dynamic_ice_loss_sensitivity_from_cumBMB.csv')

/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_41426/385205345.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.377117572882899e-06' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_dils_new.loc[i,key]= dic_all_dils[key][exp][model].copy()
/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_41426/385205345.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.901051598452382e-06' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_dils_new.loc[i,key]= dic_all_dils[key][exp][model].copy()
/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_41426/385205345.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.168661556540659e-06

In [12]:
df_dils_new

,Experiment,Model,Path,AIS,Amundsen,Ross,FilchnerRonne,Aurora,Wilkes,WestAntarctica,EastAntarctica,Peninsula
0,expAE02,DC_ISSM,/Volumes/FMac/computing/data/antarctica/ismip6...,0.000003,0.000004,0.000002,0.000004,0.000005,0.000005,0.000004,0.000003,-3.112770e-07
1,expAE02,DOE_MALI_4km,/Volumes/FMac/computing/data/antarctica/ismip6...,0.000009,0.000017,0.000005,0.000008,0.000014,0.000020,0.000009,0.000008,7.253235e-06
2,expAE02,DOE_MALI_8km_Ant95,/Volumes/FMac/computing/data/antarctica/ismip6...,0.000011,0.000014,0.000008,0.000009,0.000016,0.000026,0.000012,0.000010,6.605011e-06
3,expAE02,DOE_MALI_8km_AntMean,/Volumes/FMac/computing/data/antarctica/ismip6...,0.000010,0.000015,0.000007,0.000009,0.000015,0.000025,0.000011,0.000009,6.659005e-06
4,expAE02,IGE_ElmerIce,/Volumes/FMac/computing/data/antarctica/ismip6...,0.000003,0.000003,0.000001,0.000003,0.000005,0.000008,0.000003,0.000003,-1.106049e-06
...,...,...,...,...,...,...,...,...,...,...,...,...
167,expAE05,VUW_PISM2,/Volumes/FMac/computing/data/antarctica/ismip6...,0.000014,0.000030,0.000010,0.000010,0.000014,0.000022,0.000015,0.000012,8.224059e-06
168,expAE05,VUW_PISM2_s1,/Volumes/FMac/computing/data/antarctica/ismip6...,0.000015,0.000034,0.000012,0.000011,0.000015,0.000024,0.000015,0.000014,1.018034e-05
169,expAE05,VUW_PISM2_s2,/Volumes/FMac/computing/data/antarctica/ismip6...,0.000013,0.000026,0.000009,0.000010,0.000014,0.000019,0.000015,0.000011,6.799766e-06
170,expAE05,VUW_PISM2_s3,/Volumes/FMac/computing/data/antarctica/ismip6...,0.000011,0.000023,0.000007,0.000009,0.000011,0.000019,0.000011,0.000011,5.342969e-06


In [13]:
## save dils intercept to csv
df_dils_new_intercept = df_dils.copy()
df_dils_new_intercept=df_dils_new_intercept.drop(df_dils_new_intercept.columns[3:21],axis=1)
df_dils_new_intercept=df_dils_new_intercept.drop(df_dils_new_intercept.columns[4:7],axis=1)
for key in dic_area_of_interest.keys():
    df_dils_new_intercept[key]=0

for i in range (df_dils_new_intercept.shape[0]):
    model = df_dils_new_intercept.iloc[i].Model
    exp = df_dils_new_intercept.iloc[i].Experiment
    for j,key in enumerate(dic_area_of_interest.keys()):
        df_dils_new_intercept.loc[i,key]= dic_all_dils_intercept[key][exp][model].copy()
df_dils_new_intercept.to_csv(pth_table + '/linear_dynamic_ice_loss_sensitivity_from_cumBMB_intercept.csv')

/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_41426/82476367.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-15031.049409540377' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_dils_new_intercept.loc[i,key]= dic_all_dils_intercept[key][exp][model].copy()
/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_41426/82476367.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-8353.61937661608' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_dils_new_intercept.loc[i,key]= dic_all_dils_intercept[key][exp][model].copy()
/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_41426/82476367.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of panda

In [14]:
## save dils prmse to csv
df_err = df_dils.copy()
df_err=df_err.drop(df_err.columns[3:21],axis=1)
df_err=df_err.drop(df_err.columns[4:7],axis=1)
for key in dic_area_of_interest.keys():
    df_err[key]=0

for i in range (df_err.shape[0]):
    model = df_err.iloc[i].Model
    exp = df_err.iloc[i].Experiment
    for j,key in enumerate(dic_area_of_interest.keys()):
        df_err.loc[i,key]= dic_all_err[key][exp][model].copy()
df_err.to_csv(pth_table + '/prmse_recalc_linear_dynamic_ice_loss_sensitivity_from_cumBMB.csv')

/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_41426/60121327.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.698870691657464' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_err.loc[i,key]= dic_all_err[key][exp][model].copy()
/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_41426/60121327.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.6756489876033843' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_err.loc[i,key]= dic_all_err[key][exp][model].copy()
/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_41426/60121327.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.7319539299099077' has dtype incompatible 